In [1]:
!pip install -q "transformers>=4.51,<4.56" "peft>=0.13,<0.15" "trl>=0.12,<0.14" "bitsandbytes>=0.43.3,<0.45" "datasets>=2.20,<3.0" "accelerate>=1.3,<2" rouge-score bert-score

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
conda 23.10.0 requires ruamel-yaml<0.18,>=0.11.14, but you have ruamel-yaml 0.18.5 which is incompatible.


In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

MODEL_NAME = "Qwen/Qwen3-4B"
MAX_SEQ_LENGTH = 6454
LORA_RANK = 8

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "right"

enable_thinking = False

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [2]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_RANK,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 2,949,120 || all params: 4,025,417,216 || trainable%: 0.0733


In [3]:
import pandas as pd
from datasets import Dataset

train_df = pd.read_json("../data/preprocessed/train.jsonl", lines=True)
test_df  = pd.read_json("../data/preprocessed/test.jsonl",  lines=True)

print(f"Train: {len(train_df)} | Test: {len(test_df)}")
print(f"Columns: {list(train_df.columns)}")
train_df.head()

Train: 45 | Test: 10
Columns: ['system_prompt', 'prompt', 'ideal_response']


,system_prompt,prompt,ideal_response
0,You are a professional cosmetic consultant.\nY...,Составь персональную косметологическую консуль...,---\n\n# ПЕРСОНАЛЬНЫЙ УХОД: ЖИРНАЯ С АКНЕ\n\n#...
1,You are a professional cosmetic consultant.\nY...,Составь персональную косметологическую консуль...,---\n\n# ПЕРСОНАЛЬНЫЙ УХОД: КОМБИНИРОВАННАЯ (T...
2,You are a professional cosmetic consultant.\nY...,Составь персональную косметологическую консуль...,"---\n\n# ПЕРСОНАЛЬНЫЙ УХОД: СУХАЯ, ЗРЕЛАЯ КОЖА..."
3,You are a professional cosmetic consultant.\nY...,Составь персональную косметологическую консуль...,---\n\n# ПЕРСОНАЛЬНЫЙ УХОД: ЧУВСТВИТЕЛЬНАЯ КОЖ...
4,You are a professional cosmetic consultant.\nY...,Составь персональную косметологическую консуль...,---\n\n# ПЕРСОНАЛЬНЫЙ УХОД: ЖИРНАЯ КОЖА С ПОСТ...


In [4]:
def format_chat(row):
    messages = [
        {"role": "system", "content": row["system_prompt"]},
        {"role": "user", "content": row["prompt"]},
        {"role": "assistant", "content": row["ideal_response"]},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False, enable_thinking=enable_thinking)


train_df["text"] = train_df.apply(format_chat, axis=1)
dataset = Dataset.from_pandas(train_df[["text"]])

print(f"Example length (chars): {len(dataset[0]['text'])}")
print(dataset[0]["text"][:500])

Example length (chars): 11665
<|im_start|>system
You are a professional cosmetic consultant.
You are given a client profile and a pre-selected list of suitable products.
Write a detailed personalized consultation explaining each product, how to use it, and why it suits this client.
All listed products must appear in your response.

Client profile:
Age: 22
Skin type: Жирная с акне
Concerns: Акне, расширенные поры, себорегуляция
Budget: low
Allergies: nan
Values: Баланс (качество + цена)
Experience: Начинающий

Products for th


In [5]:
lengths = [len(tokenizer.encode(t)) for t in train_df["text"]]
print(f"Min: {min(lengths)}, Max: {max(lengths)}, Mean: {sum(lengths)//len(lengths)}")
print(f"MAX_SEQ_LENGTH: {MAX_SEQ_LENGTH}")
print(f"Примеров влезающих полностью: {sum(1 for l in lengths if l <= MAX_SEQ_LENGTH)}/{len(lengths)}")

Min: 3210, Max: 6454, Mean: 4634
MAX_SEQ_LENGTH: 6454
Примеров влезающих полностью: 45/45


In [6]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        packing=True,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        seed=42,
        output_dir="outputs",
        gradient_checkpointing=True,
    ),
)

In [ ]:
trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


In [ ]:
import matplotlib.pyplot as plt

logs = pd.DataFrame(trainer.state.log_history)
train_logs = logs[logs["loss"].notna()] if "loss" in logs.columns else logs

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(train_logs["step"], train_logs["loss"], marker="o", color="#d62728")
axes[0].set_title("Training loss")
axes[0].set_xlabel("step"); axes[0].set_ylabel("loss")
axes[0].grid(alpha=0.3)

if "learning_rate" in train_logs.columns:
    axes[1].plot(train_logs["step"], train_logs["learning_rate"], marker="o", color="#1f77b4")
    axes[1].set_title("Learning rate")
    axes[1].set_xlabel("step"); axes[1].set_ylabel("lr")
    axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
model.save_pretrained("koyash-qwen3-4b-lora")
tokenizer.save_pretrained("koyash-qwen3-4b-lora")

In [ ]:
from peft import AutoPeftModelForCausalLM

merged_model = AutoPeftModelForCausalLM.from_pretrained(
    "koyash-qwen3-4b-lora",
    torch_dtype=torch.float16,
    device_map="auto",
)
merged_model = merged_model.merge_and_unload()
merged_model.save_pretrained("koyash-qwen3-4b-merged")
tokenizer.save_pretrained("koyash-qwen3-4b-merged")